# Here we process blocks 1 and 2 to get static and atmospheric information

In [1]:
from irrigator.config import load_region_config, load_parcel_config
from irrigator.static_layers import get_soil_profile, get_terrain_params
from irrigator.static_layers.terrain import get_era5_elevation
from irrigator.atmospheric.era5_processor import process_era5_to_daily, save_daily, load_daily
from irrigator.atmospheric.forcing import extract_parcel_forcing
from irrigator.ingestion.cds_client import open_era5_land
import os

In [2]:
PARENT = str(os.path.dirname(os.getcwd()))

cfg = load_region_config(PARENT + "/configs/dordogne.yaml")
parcel = load_parcel_config(PARENT + "/configs/parcels/example.yaml")


In [3]:

# Block 1: static layers
soil = get_soil_profile(cfg, parcel)
terrain = get_terrain_params(cfg, parcel)
terrain.era5_elevation_m = get_era5_elevation(cfg, parcel)


In [4]:

# Block 2: atmospheric forcing
# First time: process hourly → daily (slow, do once)
era5_hourly = open_era5_land(cfg)
era5_daily = process_era5_to_daily(era5_hourly)
save_daily(era5_daily, cfg)


/home/mbaldacchino/code/IrriGator/src/irrigator/atmospheric/era5_processor.py:153: SerializationWarning: variable expver has data in the form of a dask array with dtype=object, which means it is being loaded into memory to determine a data type that can be safely stored on disk. To avoid this, coerce this variable to a fixed-size dtype with astype() before saving it.
  ds.to_netcdf(out_path)


In [5]:
# After that: just load
era5_daily = load_daily(cfg)

# Extract forcing at parcel (with downscaling)
forcing = extract_parcel_forcing(era5_daily, parcel, terrain)
df = forcing.to_dataframe()
df.head()


,t_min,t_max,t_mean,dewpoint,wind_speed_2m,pressure_kpa,rs_mj,precip_mm
date,,,,,,,,
1999-12-31,NaN,NaN,NaN,NaN,NaN,NaN,1.000361,8.516731
2000-01-01,5.012279,6.407787,5.679180,5.611278,0.618926,98.078033,0.903925,0.795543
2000-01-02,3.792736,7.660747,5.858501,4.469097,1.016541,98.221100,3.121260,0.043456
2000-01-03,1.761242,7.406932,4.191814,2.246135,1.742059,98.120651,5.852152,0.003600
2000-01-04,2.283764,7.584606,4.948162,3.728740,2.225215,97.914948,4.847137,0.783303
